In [1]:
import pandas as pd
df_menu = pd.read_excel(r'C:\Users\YangTeng\Downloads\CARID_menu.xlsx', header=0)
list_url = df_menu['Url'].to_list()

print('总数量：' + str(len(list_url)))
list_url

总数量：791


['https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm4055.html?singleid=3770807666',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm9313.html?singleid=3770802651',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm9353.html?singleid=3770801692',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm9313.html?singleid=3770753716',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm9318.html?singleid=3447406941',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm9309.html?singleid=2926659524',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm9242.html?singleid=2926659481',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm9299.html?singleid=2926659432',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm9215.html?singleid=2926658444',
 'https://www.carid.com/vdo/hvac-blower-motor-with-wheel-mpn-pm129.html?singleid=2926651525',
 'https://www.carid.com/vdo/hvac-blower-motor-with-

In [2]:
df = pd.DataFrame(columns=['Url',
                           'Title',
                           'mpn',
                           'Brand',
                           'Part_Number',
                           'UPC',
                           'Vehicle',
                           'Src',
                           'status'])

df

,Url,Title,mpn,Brand,Part_Number,UPC,Vehicle,Src,status


In [3]:
from selenium.webdriver import ChromeOptions
option = ChromeOptions()
option.add_experimental_option('useAutomationExtension', False)
option.add_argument('blink-settings=imagesEnabled=false')
option.add_argument('--disable-blink-features=AutomationControlled')
option.add_argument('--disable-dev-shm-usage')
option.add_argument('--disable-gpu')
option.add_experimental_option('excludeSwitches', ['enable-automation'])
# option.add_argument('--headless')
option.add_argument('--no-sandbox')
option.add_argument('--proxy-server=http://u431.kdltps.com:15818')
option.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36')
option.page_load_strategy = 'eager'

In [4]:
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
browser = Chrome(service=Service(r'C:\Users\YangTeng\Desktop\venv\chromedriver'), options=option)

browser.maximize_window()

In [5]:
import time
i = 4

browser.get(list_url[i])

time.sleep(5)

In [6]:
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By

wait = WebDriverWait(browser, 60)
wait.until(EC.presence_of_element_located((By.XPATH, '//div[@class="prod-title js-prod-title"]/h1[@class="name"]')))
title = browser.find_element(by=By.XPATH, value='//div[@class="prod-title js-prod-title"]/h1[@class="name"]').text

title

'VDO® - HVAC Blower Motor with Wheel'

In [7]:
list_row = browser.find_element(by=By.XPATH, value='//div[@class="prod-title js-prod-title"]').text.split('|')
mpn = ''
for row in list_row:
    if row.startswith('Item # mpn'):
        mpn = row.split('Item # mpn')[1]

mpn

'4689752193'

In [10]:
wait.until(EC.presence_of_element_located((By.XPATH, '//div[@class="prod-offer"]')))
list_offer = browser.find_elements(by=By.XPATH, value='//div[@class="prod-offer"]/div[@class="prod-offer-item"]')

list_offer

[<selenium.webdriver.remote.webelement.WebElement (session="ac1b79db209fc9107c77aea02e7f5673", element="E8BB43AFE14419B004CB3DA8CA824683_element_54")>,
 <selenium.webdriver.remote.webelement.WebElement (session="ac1b79db209fc9107c77aea02e7f5673", element="E8BB43AFE14419B004CB3DA8CA824683_element_55")>,
 <selenium.webdriver.remote.webelement.WebElement (session="ac1b79db209fc9107c77aea02e7f5673", element="E8BB43AFE14419B004CB3DA8CA824683_element_56")>,
 <selenium.webdriver.remote.webelement.WebElement (session="ac1b79db209fc9107c77aea02e7f5673", element="E8BB43AFE14419B004CB3DA8CA824683_element_57")>]

In [11]:
brand = ''
part_number = ''
upc = ''
for offer in list_offer:
    if offer.text.startswith('Brand:'):
        brand = offer.text.split('Brand:')[1]
    elif offer.text.startswith('Part Number:'):
        part_number = offer.text.split('Part Number:')[1]
    elif offer.text.startswith('UPC:'):
        upc = offer.text.split('UPC:')[1]
        
brand, part_number, upc

('VDO', 'PM9318', '053674093182')

In [12]:
js = '''let list_expand = document.querySelectorAll('div[class="mmy-row"]');
for(let i=0; i<list_expand.length; i++) {
        list_expand[i].getElementsByClassName('mmy-row-expand')[0].click();
}'''

browser.execute_script(js)

In [13]:
wait.until(EC.presence_of_element_located((By.XPATH, '//div[@class="prod-section-content"]')))
list_row = browser.find_elements(by=By.XPATH, value='//div[@class="prod-section-content"]')

list_vehicle = []
for row in list_row:
    if row.text.startswith('Vehicle Fitment'):
        list_make = row.find_elements(by=By.XPATH, value='./descendant::div[@class="mmy-row-title pointer js-mmy-row-expand"]')
        for make in list_make:
            list_vehicle += [make.text + ' ' + model_year.text for model_year in make.find_elements(by=By.XPATH, value='./following-sibling::div[@class="mmy-row-description"]/descendant::div[@class="mmy-subrow-title"]')]
            
vehicle = '\n'.join(list_vehicle)

vehicle

'Nissan Altima 2005-2006\nNissan Maxima 2003-2007'

In [14]:
wait.until(EC.presence_of_element_located((By.XPATH, '//a[@class="product-image-main"]')))
src = browser.find_element(by=By.XPATH, value='//a[@class="product-image-main"]').get_attribute('href')

src

'https://images.carid.com/vdo/items/pm9318.jpg'

In [15]:
df_temp = pd.DataFrame([{'Url': list_url[i],
                         'Title': title,
                         'mpn': mpn,
                         'Brand': brand,
                         'Part_Number': part_number,
                         'UPC': upc,
                         'Vehicle': vehicle,
                         'Src': src,
                         'status': 'ok'}])

df_temp

,Url,Title,mpn,Brand,Part_Number,UPC,Vehicle,Src,status
0,https://www.carid.com/vdo/hvac-blower-motor-wi...,VDO® - HVAC Blower Motor with Wheel,4689752193,VDO,PM9318,053674093182,Nissan Altima 2005-2006\nNissan Maxima 2003-2007,https://images.carid.com/vdo/items/pm9318.jpg,ok


In [16]:
wait.until(EC.presence_of_element_located((By.XPATH, '//table[@class="spec-stripped-table"]')))
list_row = [row.text for row in browser.find_elements(by=By.XPATH, value='//table[@class="spec-stripped-table"]/tbody/tr/td')]

start = ''
for row in list_row:
    if ':' in row:
        name_value = row.split(': ')
        df_temp[(start + ' ' + name_value[0]).strip().lower()] = ': '.join(name_value[1:])
    else:
        start = row
        
browser.quit()

df = pd.concat([df, df_temp], ignore_index=True).fillna('')

df

,Url,Title,mpn,Brand,Part_Number,UPC,Vehicle,Src,status,voltage,...,mounting hardware included,terminal quantity,terminal gender,mounting method,connector gender,connector type,wheel diameter,flanged end,vented,shaft type
0,https://www.carid.com/vdo/hvac-blower-motor-wi...,VDO® - HVAC Blower Motor with Wheel,4689752193,VDO,PM9318,053674093182,Nissan Altima 2005-2006\nNissan Maxima 2003-2007,https://images.carid.com/vdo/items/pm9318.jpg,ok,12V,...,No,2,Male,Flange Mounted,Female,Plug,6.000 in,Yes,Yes,Single Shaft
